# PICO 2 W AI SMART CLASSROOM
## ① 오늘의 질문
AI가 판단하는 규칙은 사람이 정한 것이 아니라, 학습 데이터로부터 모델이 찾은 것이다.
다만 무엇을 정답이라고 부를지, 어떤 데이터를 모을지, 어떤 특성을 쓸지는 사람이 정한다. 모델의 판단은 그 선택과 데이터의 한계를 가진다.

feature 순서: light(raw), temperature(℃), humidity(%RH). label은 사람이 합의한 행동입니다. state를 정답으로 복사하지 않습니다. VENTILATE는 CO₂나 공기질 측정이 아니며 GOOD도 안전 보장이 아닙니다.

실제 같은 키트·배선·저항으로 모은 자료만 사용합니다. 정확도는 실행 후 측정합니다. 두 상태 예비 실습은 네 상태 최종 완료와 다릅니다.

수정할 것: CSV 선택, 해석 문장, 실험 가설. 고급 변환 셀의 feature 순서·전처리는 바꾸지 않습니다.

In [ ]:
# ② 스마트폰 CSV를 PC로 옮긴 뒤 업로드 (USB 또는 학교 허용 방식)
from google.colab import files
uploaded = files.upload()
assert uploaded, '실제 classroom CSV를 선택하세요.'


In [ ]:
# ③ 표·단위·품질 검사. 오류를 조용히 제외하지 않습니다.
"""Strict CSV quality gate. No silent error-row dropping; no synthetic training data."""
import io
import hashlib
import re
import pandas as pd

CLASSES = ['GOOD','DARK','VENTILATE','HOT_HUMID']
HEADER = ['schema_version','source','session_id','sample_index','light','temperature','humidity','label']
UUID = re.compile(r'^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$')

def inspect_csvs(uploaded):
    frames, issues, hashes = [], [], []
    for number, raw in enumerate(uploaded.values(),1):
        hashes.append(hashlib.sha256(raw).hexdigest())
        try:
            frame = pd.read_csv(io.BytesIO(raw),dtype=str,keep_default_na=False)
        except Exception as exc:
            issues.append(f'파일 {number}: CSV 읽기 실패 ({type(exc).__name__})'); continue
        if list(frame.columns) != HEADER:
            issues.append(f'파일 {number}: v1 헤더 불일치'); continue
        for index,row in frame.iterrows():
            reasons=[]
            if row.schema_version != '1': reasons.append('버전')
            if row.source != 'real': reasons.append('실측 source 아님')
            if not UUID.fullmatch(row.session_id): reasons.append('회차 UUID')
            if row.label not in CLASSES: reasons.append('label')
            for name,lo,hi,integer in [('sample_index',1,3000,True),('light',0,65535,True),('temperature',0,50,False),('humidity',0,100,False)]:
                value=row[name]
                if not re.fullmatch(r'\d+' if integer else r'\d+(\.\d)?',value): reasons.append(name+' 형식');continue
                if not lo<=float(value)<=hi: reasons.append(name+' 범위')
            if reasons: issues.append(f'파일 {number}, {index+2}행: '+', '.join(reasons))
        frames.append(frame)
    if not frames: return pd.DataFrame(columns=HEADER),issues or ['자료 없음'],0,hashes
    data=pd.concat(frames,ignore_index=True)
    for session,group in data.groupby('session_id'):
        if group.label.nunique()!=1: issues.append('회차 label 혼재: '+session)
    keys=['session_id','sample_index']
    for _,group in data.groupby(keys):
        if len(group.drop_duplicates())>1: issues.append('같은 회차/번호의 서로 다른 값')
    duplicate_count=int(data.duplicated(keys).sum())
    data=data.drop_duplicates(keys)
    if not issues:
        for name in ['sample_index','light','temperature','humidity']:
            data[name]=pd.to_numeric(data[name])
    return data,issues,duplicate_count,hashes

def require_ready(df):
    if len(df)<80: raise ValueError('총 80행 이상 실제 자료 필요')
    sessions=df.groupby('session_id').agg(label=('label','first'),labels=('label','nunique'),rows=('label','size'))
    if not (sessions.labels==1).all(): raise ValueError('회차 label 혼재')
    if not (sessions.rows>=5).all(): raise ValueError('모든 회차에 유효 실측 5행 이상 필요')
    counts=sessions.label.value_counts().reindex(CLASSES,fill_value=0)
    if (counts<4).any(): raise ValueError('네 라벨마다 독립 회차 4개 이상 필요: '+str(counts.to_dict()))
    return sessions.reset_index()[['session_id','label']]

df, issues, duplicates, csv_hashes = inspect_csvs(uploaded)
print('중복 업로드 제거:', duplicates, '오류:', len(issues))
for issue in issues: print(issue)
assert not issues, '원본 CSV의 오류를 수정하고 다시 업로드하세요. 학습을 중단합니다.'
display(df.head())
print('단위: raw / ℃ / %RH. 온습도 소수 최대 한 자리.')


In [ ]:
# ④ 실제 라벨·회차 분포와 준비 상태
sessions = require_ready(df)
display(sessions.label.value_counts().reindex(CLASSES, fill_value=0))
display(df.label.value_counts().reindex(CLASSES, fill_value=0))
sessions.label.value_counts().reindex(CLASSES, fill_value=0).plot.bar(title='Independent sessions by label')
print('회차 버튼만 다시 누른 것은 독립 자료가 아닙니다. 다른 조건/시간을 확인하세요.')


In [ ]:
# ⑤ 회차 단위 train_test_split: 원시 행 무작위 분리 금지
from sklearn.model_selection import train_test_split
train_sessions, test_sessions = train_test_split(sessions, test_size=0.25, random_state=42, stratify=sessions.label)
train = df[df.session_id.isin(train_sessions.session_id)].copy()
test = df[df.session_id.isin(test_sessions.session_id)].copy()
assert set(train.session_id).isdisjoint(set(test.session_id))
print('회차 교집합 0 / 행수:', len(train), len(test))


In [ ]:
# ⑥ 학습 / 교사용 고급 셀: 같은 정수 전처리와 tree_ 변환 원본
"""Single-output sklearn decision trees, exactly three integer-scaled features only."""
import hashlib
import json
import math
import platform
from pathlib import Path
import numpy as np
import sklearn

FEATURES = ['light', 'temperature', 'humidity']
MODEL_FEATURES = ['light_raw', 'temperature_x10', 'humidity_x10']
PREPROCESS = 'raw-int,tenth-int,tenth-int-v1'

def features_for_model(frame):
    out = frame[FEATURES].copy()
    out['light'] = out['light'].astype(int)
    for name in FEATURES[1:]:
        out[name] = out[name].map(lambda x: int(round(float(x)*10)))
    out.columns = MODEL_FEATURES
    return out

def export_pico_model(clf, ranges=None):
    if clf.n_outputs_ != 1 or clf.n_features_in_ != 3:
        raise ValueError('Only single-output, three-feature classifiers are supported')
    if hasattr(clf, 'feature_names_in_') and list(clf.feature_names_in_) != MODEL_FEATURES:
        raise ValueError('Unexpected feature order')
    if clf.get_depth() > 3:
        raise ValueError('Course model depth must be <= 3')
    tree = clf.tree_
    lines = ['# Automatically generated from fitted model.tree_. Do not edit thresholds.',
             'FEATURES = '+repr(tuple(FEATURES)), 'PREPROCESS = '+repr(PREPROCESS),
             'TRAINING_RANGES = '+repr(ranges),
             'def predict(light, temperature, humidity):',
             '    light_raw = int(light)',
             '    temperature_x10 = int(round(temperature * 10))',
             '    humidity_x10 = int(round(humidity * 10))']
    def walk(node, depth):
        pad = '    '*depth
        left, right = int(tree.children_left[node]), int(tree.children_right[node])
        if left == right:
            label = str(clf.classes_[int(np.argmax(tree.value[node][0]))])
            lines.append(pad+'return '+repr(label))
            return
        feature = MODEL_FEATURES[int(tree.feature[node])]
        cut = math.floor(float(tree.threshold[node]))
        lines.append(pad+f'if {feature} <= {cut}:')
        walk(left, depth+1)
        lines.append(pad+'else:')
        walk(right, depth+1)
    walk(0, 1)
    body = '\n'.join(lines)+'\n'
    model_id = 'dt-'+hashlib.sha256(body.encode()).hexdigest()[:12]
    return 'MODEL_ID = '+repr(model_id)+'\n'+body, model_id

def boundary_probes(clf, domain=((0,65535),(0,500),(0,1000))):
    """Visit each split with all ancestor path constraints; never test unreachable probes."""
    probes = []
    tree = clf.tree_
    def visit(node, bounds):
        left, right = int(tree.children_left[node]), int(tree.children_right[node])
        if left == right:
            return
        feature = int(tree.feature[node]); cut = math.floor(float(tree.threshold[node]))
        for value in (cut-1, cut, cut+1):
            if bounds[feature][0] <= value <= bounds[feature][1]:
                point = [(lo+hi)//2 for lo,hi in bounds]
                point[feature] = value
                probes.append(point)
        for child, is_left in ((left,True),(right,False)):
            next_bounds = list(bounds)
            lo, hi = bounds[feature]
            next_bounds[feature] = (lo,min(hi,cut)) if is_left else (max(lo,cut+1),hi)
            if all(a<=b for a,b in next_bounds):
                visit(child,next_bounds)
    visit(0,list(domain))
    return probes

def validate_conversion(clf, source, raw_rows, domain=((0,65535),(0,500),(0,1000))):
    # Execute only source generated by export_pico_model in this process.
    namespace = {}
    exec(compile(source, '<generated-classroom-model>', 'exec'), namespace)
    rng = np.random.default_rng(42)
    synthetic = np.column_stack([rng.integers(lo, hi+1, 1000) for lo,hi in domain]).tolist()
    probes = boundary_probes(clf, domain)
    real = [[int(l),int(round(t*10)),int(round(h*10))] for l,t,h in raw_rows]
    scaled = real+synthetic+probes
    # Inputs only; fixtures never enter fit or measured accuracy.
    import pandas as pd
    expected = clf.predict(pd.DataFrame(scaled, columns=MODEL_FEATURES) if hasattr(clf,'feature_names_in_') else np.array(scaled))
    tests = []
    for point,label in zip(scaled,expected):
        values = [point[0],point[1]/10,point[2]/10]
        actual = namespace['predict'](*values)
        if actual != label:
            raise AssertionError((values,label,actual))
        tests.append({'input':values,'expected':str(label)})
    return tests

def write_artifacts(clf, train, test, csv_hashes, metrics, directory='.'):
    ranges = [(float(train[f].min()),float(train[f].max())) for f in FEATURES]
    source, model_id = export_pico_model(clf, ranges)
    raw_rows = list(train[FEATURES].itertuples(index=False,name=None))+list(test[FEATURES].itertuples(index=False,name=None))
    tests = validate_conversion(clf,source,raw_rows)
    metadata = dict(model_id=model_id, source_sha256=hashlib.sha256(source.encode()).hexdigest(),
        features=FEATURES, units=['raw','C','%RH'], preprocess=PREPROCESS,
        classes=[str(c) for c in clf.classes_], max_depth=clf.max_depth,
        min_samples_leaf=clf.min_samples_leaf, random_state=clf.random_state,
        train_count=len(train), test_count=len(test),
        train_class_counts={str(k):int(v) for k,v in train.label.value_counts().items()},
        test_class_counts={str(k):int(v) for k,v in test.label.value_counts().items()},
        train_sessions=sorted(train.session_id.unique()),test_sessions=sorted(test.session_id.unique()),
        csv_sha256=csv_hashes, python=platform.python_version(),sklearn=sklearn.__version__,numpy=np.__version__,
        metrics=metrics,training_ranges=dict(zip(FEATURES,ranges)),hardware_status='NOT_RUN')
    out = Path(directory); out.mkdir(parents=True,exist_ok=True)
    (out/'model.py').write_bytes(source.encode('utf-8'))  # Preserve the hashed LF bytes on Windows too.
    (out/'model-meta.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
    (out/'model-tests.json').write_text(json.dumps({'model_id':model_id,'tests':tests},ensure_ascii=False),encoding='utf-8')
    return metadata

from sklearn.tree import DecisionTreeClassifier, export_text
X_train, X_test = features_for_model(train), features_for_model(test)
model = DecisionTreeClassifier(max_depth=3, min_samples_leaf=3, random_state=42)
model.fit(X_train, train.label)


In [ ]:
# ⑦ 평가: test로 fit하거나 좋은 결과가 나올 때까지 seed를 바꾸지 않습니다.
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.dummy import DummyClassifier
pred = model.predict(X_test)
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, train.label)
metrics = dict(train_accuracy=accuracy_score(train.label, model.predict(X_train)), accuracy=accuracy_score(test.label,pred), baseline=accuracy_score(test.label,baseline.predict(X_test)), confusion_matrix=confusion_matrix(test.label,pred,labels=CLASSES).tolist(), confusion_labels=CLASSES)
print(metrics)
display(pd.DataFrame(metrics['confusion_matrix'],index=CLASSES,columns=CLASSES))
display(test.loc[pred != test.label, ['light','temperature','humidity','label']])
interpretation = '실행 후 오분류 조건과 다수 클래스 기준을 비교해 직접 적으세요.'
print(interpretation)


In [ ]:
# ⑧ 경로 읽기. export_text는 설명 전용이며 다시 파싱하지 않습니다.
print(export_text(model, feature_names=list(X_train.columns), decimals=1))
print('temperature_x10 <= 275라면 27.5℃ 이하. 숫자는 학습에서 얻은 것입니다.')
print('테스트를 반복 튜닝에 사용하지 마세요. 필요하면 train 내부에 validation 회차를 만드세요.')


In [ ]:
# ⑨ 자동 변환과 동등성. 학습/평가 전체 + 변환 검사 전용 합성 1000개 + 경로별 cut±1
metadata = write_artifacts(model, train, test, csv_hashes, metrics)
print('CPython 동등성 PASS / model_id:', metadata['model_id'])
print('합성 입력은 변환 검사 전용이며 fit/accuracy에 사용하지 않았습니다. MicroPython 실기는 NOT_RUN입니다.')


In [ ]:
# ⑩ 파일 다운로드와 새 자료 검증
for filename in ['model.py','model-meta.json','model-tests.json']:
    files.download(filename)
print('model.py: Pico 추론 코드 / meta: 출처·평가·학습 범위 / tests: 실기 동등성 입력')
print('Thonny로 Pico 루트에 model.py와 model-tests.json을 저장하고 model_selftest.py를 실행하세요.')
print('config.py INFERENCE_MODE=True 후 재시작. 학습 범위 밖이면 OUT_OF_DOMAIN과 출력 정지.')
print('7~8차시는 다른 시간·조건의 새 센서 입력 10회 이상을 별도 검증하세요. 범위 안도 정답 보장은 아닙니다.')
print('새 검증 결과는 학습 CSV와 분리해 기록하고 실제 오분류·센서 한계·다음 수집 조건을 3문장으로 설명하세요.')
